In [1]:
import numpy as np
import pandas as pd

In [2]:
START = "1990-01-01"
TAUS = np.arange(1.0, 30.0 + 1e-9, 0.5)
DELTA = 1.0 / 252.0
 
PARAM_COLS = ["BETA0", "BETA1", "BETA2", "BETA3", "TAU1", "TAU2"]

In [3]:
df = pd.read_csv('feds200628.csv', skiprows=9)
print(df.tail())

             Date     BETA0     BETA1       BETA2       BETA3  SVEN1F01  \
16963  2026-06-22  0.001363  4.088206 -444.258444  455.034384    4.2250   
16964  2026-06-23  0.002329  4.051780 -456.391376  467.196656    4.1957   
16965  2026-06-24  0.000879  4.054607 -463.160934  473.621009    4.1225   
16966  2026-06-25  0.000917  4.054669 -541.698465  552.113208    4.0956   
16967  2026-06-26  0.000726  4.038977 -527.729559  538.167567    4.0623   

       SVEN1F04  SVEN1F09  SVENF01  SVENF02  ...  SVENY23  SVENY24  SVENY25  \
16963    4.5461    5.1519   4.1430   4.2233  ...   5.0786   5.0939   5.1059   
16964    4.5321    5.1569   4.1119   4.1971  ...   5.0757   5.0907   5.1023   
16965    4.4062    5.0620   4.0593   4.1087  ...   4.9964   5.0110   5.0217   
16966    4.3766    5.0753   4.0388   4.0778  ...   5.0019   5.0156   5.0253   
16967    4.3472    5.0875   4.0096   4.0428  ...   5.0132   5.0274   5.0373   

       SVENY26  SVENY27  SVENY28  SVENY29  SVENY30       TAU1       TAU2  

In [4]:
def svensson_forward(params: pd.DataFrame, tau: np.ndarray) -> np.ndarray:
    """Instantaneous forward f(t, t+tau) in percent.
 
    params: rows = dates. tau: shape (K,) or (J, K) for per-row grids.
    """
    tau = np.atleast_2d(np.asarray(tau, dtype=float))
    b0, b1, b2, b3, t1, t2 = (params[c].to_numpy()[:, None] for c in PARAM_COLS)
    e1 = np.exp(-tau / t1)
    e2 = np.exp(-tau / t2)
    return b0 + b1 * e1 + b2 * (tau / t1) * e1 + b3 * (tau / t2) * e2


In [5]:
def load_gsw() -> pd.DataFrame:
    df = pd.read_csv('feds200628.csv', skiprows=9)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.set_index("Date").sort_index()
    df = df[PARAM_COLS].apply(pd.to_numeric, errors="coerce")
    return df.dropna()

In [6]:
def build_panels(df: pd.DataFrame, taus: np.ndarray, start: str):
    df = df.loc[start:]
    p_now, p_next = df.iloc[:-1], df.iloc[1:]
 
    F = svensson_forward(p_now, taus) / 100.0                 # f(t_j, t_j+tau_k)
    Fd = svensson_forward(p_next, taus - DELTA) / 100.0       # f(t_j+delta, t_j+tau_k)
 
    return df.index[:-1].to_numpy(), F, Fd

In [7]:
df = load_gsw()
dates, F, Fd = build_panels(df, TAUS, START)
print(f"J = {len(dates)} dates, K = {len(TAUS)} maturities, delta = 1/252")
np.savez("hjm_panels.npz", dates=dates, taus=TAUS, delta=DELTA, F=F, Fd=Fd)
print("saved -> hjm_panels.npz")

J = 9107 dates, K = 59 maturities, delta = 1/252
saved -> hjm_panels.npz


In [8]:
print(F)

[[0.07700877 0.07744428 0.07791269 ... 0.07731798 0.07725352 0.07719099]
 [0.07765477 0.07798887 0.07844516 ... 0.07741445 0.07735528 0.0772982 ]
 [0.077342   0.07779879 0.07828125 ... 0.0776142  0.07755018 0.07748813]
 ...
 [0.04111864 0.04151572 0.04197105 ... 0.05035034 0.04989239 0.04942262]
 [0.04059299 0.04079114 0.04108746 ... 0.04884271 0.04832636 0.04779829]
 [0.04038787 0.04052293 0.04077797 ... 0.04816656 0.04760253 0.04702733]]
